# Semantic Search (시맨틱 검색 엔진 구축)

이 튜토리얼에서는 LangChain의 문서 로더, 임베딩, 벡터 스토어를 사용하여 PDF 문서에 대한 시맨틱 검색 엔진을 구축합니다.

**시맨틱 검색(Semantic Search)** 은 키워드 매칭이 아닌 의미 기반으로 문서를 검색하는 방법입니다.
임베딩을 사용하여 텍스트를 벡터로 변환하고, 유사도 기반으로 관련 문서를 찾습니다.

**참고**: [LangChain 공식 문서 - Semantic Search](https://docs.langchain.com/oss/python/langchain/knowledge-base)

## 학습 내용
- 문서 로더 (Document Loaders)
- 텍스트 분할 (Text Splitting)
- 임베딩 (Embeddings)
- 벡터 스토어 (Vector Stores)
- 검색기 (Retrievers)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# LangSmith 추적 (선택적)
langsmith_api_key = os.getenv("LANGCHAIN_API_KEY", "")
if langsmith_api_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"

## 1. 문서 로드하기

PDF 파일을 `Document` 객체로 로드합니다.
LangChain은 다양한 문서 소스를 지원합니다: PDF, 웹페이지, 데이터베이스, API 등.

In [2]:
# pip install pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드
file_path = "./example_data/nke-10k-2023_korean.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"로드된 페이지 수: {len(docs)}")
print(f"\n첫 페이지 미리보기:\n{docs[0].page_content[:200]}...")
print(f"\n메타데이터: {docs[0].metadata}")

로드된 페이지 수: 103

첫 페이지 미리보기:
목차 
미국 
증권거래위원회(SEC) 워싱턴 D.C. 20549 
양식 10-K 
(마크 1) 
☑ 2023 년 5 월 31 일에 종료되는 회계 연도에 대한 1934 년 증권거래법 섹션 13 또는 15(D)에 따른 연례 보고서 
또는 
☐ 1934 년 증권거래법(SECURITIES EXCHANGE ACT OF 1934)의 섹션 13 또는 15(D)에 따른 ...

메타데이터: {'producer': 'Microsoft® Word Microsoft 365용', 'creator': 'Microsoft® Word Microsoft 365용', 'creationdate': '2024-12-26T10:17:40+09:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2024-12-26T10:17:40+09:00', 'source': './example_data/nke-10k-2023_korean.pdf', 'total_pages': 103, 'page': 0, 'page_label': '1'}


## 2. 텍스트 분할 (Text Splitting)

페이지 단위는 검색에 너무 큽니다. 더 작은 청크로 분할하여 정확한 검색이 가능하도록 합니다.

**중요 설정:**
- `chunk_size`: 각 청크의 최대 크기
- `chunk_overlap`: 청크 간 중첩 (문맥 유지)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 각 청크 최대 1000자
    chunk_overlap=200,    # 청크 간 200자 중첩
    add_start_index=True  # 원본 문서 내 시작 위치 저장
)

# 문서 분할
all_splits = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(all_splits)}")
print(f"\n첫 번째 청크:\n{all_splits[0].page_content[:300]}...")

분할된 청크 수: 283

첫 번째 청크:
목차 
미국 
증권거래위원회(SEC) 워싱턴 D.C. 20549 
양식 10-K 
(마크 1) 
☑ 2023 년 5 월 31 일에 종료되는 회계 연도에 대한 1934 년 증권거래법 섹션 13 또는 15(D)에 따른 연례 보고서 
또는 
☐ 1934 년 증권거래법(SECURITIES EXCHANGE ACT OF 1934)의 섹션 13 또는 15(D)에 따른 전환 보고서 
에서 로 전환 기간 동안 . 
커미션 파일 번호 1-10635 
 
주식회사 나이키 
(정관에 명시된 등록자의 정확한 이름) 
 오 레 곤 93-0584541 
 (...


## 3. 임베딩 (Embeddings)

텍스트를 숫자 벡터로 변환합니다. 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 있습니다.

In [5]:
# OpenAI 임베딩 사용 (권장)
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 또는 Hugging Face 한국어 모델 사용
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1")

# 임베딩 테스트
query_vector = embeddings.embed_query("나이키의 매출")
print(f"임베딩 벡터 차원: {len(query_vector)}")
print(f"벡터 일부: {query_vector[:5]}...")

임베딩 벡터 차원: 1536
벡터 일부: [0.048704057931900024, 0.022023988887667656, 0.02420421689748764, 0.008060373365879059, 0.02196856029331684]...


## 4. 벡터 스토어 (Vector Store)

임베딩된 문서를 저장하고 유사도 검색을 수행합니다.
여러 벡터 스토어 옵션: InMemoryVectorStore, Chroma, Pinecone, Weaviate 등

### 4.1 InMemoryVectorStore (간단한 테스트용)

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

# 인메모리 벡터 스토어 생성
vector_store = InMemoryVectorStore(embeddings)

# 문서 추가
document_ids = vector_store.add_documents(documents=all_splits)
print(f"저장된 문서 ID 수: {len(document_ids)}")

저장된 문서 ID 수: 283


### 4.2 Chroma (영구 저장)

In [7]:
from langchain_chroma import Chroma

# Chroma 벡터 스토어 생성 (로컬에 영구 저장)
vector_store = Chroma(
    collection_name="nike_10k",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"  # 로컬 저장 경로
)

# 문서 추가
document_ids = vector_store.add_documents(documents=all_splits)
print(f"Chroma에 저장된 문서 수: {len(document_ids)}")

Chroma에 저장된 문서 수: 283


## 5. 검색 수행

벡터 스토어에서 유사도 기반 검색을 수행합니다.

In [8]:
# 기본 유사도 검색
query = "나이키는 미국에 몇 개의 유통 센터를 가지고 있나요?"
results = vector_store.similarity_search(query, k=3)

print(f"검색 쿼리: {query}\n")
print("=" * 80)
for i, doc in enumerate(results, 1):
    print(f"\n[결과 {i}]")
    print(f"페이지: {doc.metadata.get('page', 'N/A')}")
    print(f"내용: {doc.page_content[:300]}...")

검색 쿼리: 나이키는 미국에 몇 개의 유통 센터를 가지고 있나요?


[결과 1]
페이지: 4
내용: Consumer) 사업부는 미국 내 다음과 같은 수의 소매점을 통해 제품을 판매합니다. 
미국 소매점 수 
NIKE 브랜드 팩토리 스토어 213  
NIKE 브랜드 인라인 매장(사원 전용 매장 포함) 74  
컨버스 점포(팩토리 점포 포함) 82  
합계 369  
미국에는 NIKE 가 8 개의 주요 유통 센터를 보유하고 있습니다. 항목 2 를 참조하십시오. properties 를 참조하십시오. 
2023 양식 10-K 2 
 
국제 시장 
2023 회계연도의 경우, 미국 외 지역 NIKE 브랜드와 컨버스 매출이 전체 매출의 약 5...

[결과 2]
페이지: 21
내용: 애널리스트의 추정치는 종종 우리의 추정치 또는 기대와 다릅니다. 당사의 영업 실적이 공개 시장 분석가 및 투자자의 추정치 또는 기대치를 밑돌면 주가가 하락할 수 
있습니다. 과거에는 증권 시장 가격이 하락한 후 NIKE 및 기타 회사를 상대로 증권 집단 소송이 제기되었습니다. 어떤 이유로든 주가가 변동하는 경우 향후 이러한 유형의 
소송에 연루될 수 있습니다. 모든 소송은 평판 손상, 상당한 비용 및 비즈니스를 성공적으로 운영하는 데 필요한 경영진의 관심과 자원의 전환을 초래할 수 있습니다. 
2023 양식 10-K 23 
 
항목...

[결과 3]
페이지: 21
내용: 전략을 구현하는 데 중점을 둔 직원들이 사용하고 있습니다. 
미국에는 NIKE 가 8 개의 주요 유통 센터를 보유하고 있습니다. 5 개는 테네시 주 멤피스 또는 그 인근에 위치하고 있으며, 그 중 2 개는 소유이고 3 개는 임대입니다. 
인디애나주 인디애나폴리스와 테네시주 데이턴에 위치한 다른 두 개의 유통 센터는 제 3 자 물류 제공업체가 임대 및 운영합니다. Converse 의 유통 센터 중 하나는 
캘리포니아 온타리오에 있으며 임대되어 있습니다. NIKE 는 미국 이외의 지역에 여러 유통 시설을 보유하고 있으며, 그 

### 5.1 유사도 점수 포함 검색

In [9]:
# 유사도 점수와 함께 검색
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

print(f"검색 쿼리: {query}\n")
print("=" * 80)
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n[결과 {i}] 유사도 점수: {score:.4f}")
    print(f"페이지: {doc.metadata.get('page', 'N/A')}")
    print(f"내용: {doc.page_content[:200]}...")

검색 쿼리: 나이키는 미국에 몇 개의 유통 센터를 가지고 있나요?


[결과 1] 유사도 점수: 1.0020
페이지: 4
내용: Consumer) 사업부는 미국 내 다음과 같은 수의 소매점을 통해 제품을 판매합니다. 
미국 소매점 수 
NIKE 브랜드 팩토리 스토어 213  
NIKE 브랜드 인라인 매장(사원 전용 매장 포함) 74  
컨버스 점포(팩토리 점포 포함) 82  
합계 369  
미국에는 NIKE 가 8 개의 주요 유통 센터를 보유하고 있습니다. 항목 2 를 참조하십시오...

[결과 2] 유사도 점수: 1.0066
페이지: 21
내용: 애널리스트의 추정치는 종종 우리의 추정치 또는 기대와 다릅니다. 당사의 영업 실적이 공개 시장 분석가 및 투자자의 추정치 또는 기대치를 밑돌면 주가가 하락할 수 
있습니다. 과거에는 증권 시장 가격이 하락한 후 NIKE 및 기타 회사를 상대로 증권 집단 소송이 제기되었습니다. 어떤 이유로든 주가가 변동하는 경우 향후 이러한 유형의 
소송에 연루될 수 있습니...

[결과 3] 유사도 점수: 1.0575
페이지: 21
내용: 전략을 구현하는 데 중점을 둔 직원들이 사용하고 있습니다. 
미국에는 NIKE 가 8 개의 주요 유통 센터를 보유하고 있습니다. 5 개는 테네시 주 멤피스 또는 그 인근에 위치하고 있으며, 그 중 2 개는 소유이고 3 개는 임대입니다. 
인디애나주 인디애나폴리스와 테네시주 데이턴에 위치한 다른 두 개의 유통 센터는 제 3 자 물류 제공업체가 임대 및 운영합...


## 6. 검색기 (Retriever)

검색기는 Runnable 인터페이스를 구현하여 표준화된 방식으로 검색을 수행합니다.
벡터 스토어의 `as_retriever()` 메서드로 검색기를 생성할 수 있습니다.

In [10]:
# 검색기 생성
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # 상위 3개 결과 반환
)

# 단일 쿼리
results = retriever.invoke(query)
print(f"검색 결과 수: {len(results)}")

# 배치 쿼리 (여러 쿼리 동시 처리)
queries = [
    "2023년 나이키의 매출은 얼마였나요?",
    "나이키의 주요 시장은 어디인가요?"
]

batch_results = retriever.batch(queries)
for i, (q, docs) in enumerate(zip(queries, batch_results), 1):
    print(f"\n[쿼리 {i}]: {q}")
    print(f"결과 수: {len(docs)}")
    if docs:
        print(f"첫 결과: {docs[0].page_content[:150]}...")

검색 결과 수: 3

[쿼리 1]: 2023년 나이키의 매출은 얼마였나요?
결과 수: 3
첫 결과: 목차 
2023 양식 10-K 54 
 
나이키, INC. 
연결손익계산서 
연말 5 월 31 일, 
 
(단위: 백만 단위, 주당 데이터 제외) 2023 2022 2021 
수익 $ 51,217  $ 46,710 $ 44,538  
매출 원가  28,925   25,...

[쿼리 2]: 나이키의 주요 시장은 어디인가요?
결과 수: 3
첫 결과: 목차 
2023 양식 10-K 54 
 
나이키, INC. 
연결손익계산서 
연말 5 월 31 일, 
 
(단위: 백만 단위, 주당 데이터 제외) 2023 2022 2021 
수익 $ 51,217  $ 46,710 $ 44,538  
매출 원가  28,925   25,...


### 6.1 다양한 검색 전략

In [11]:
# MMR (Maximum Marginal Relevance) 검색
# 유사도와 다양성을 균형있게 고려
retriever_mmr = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # lambda_mult: 다양성 비율 (0~1)
)

mmr_results = retriever_mmr.invoke(query)
print("MMR 검색 결과:")
for i, doc in enumerate(mmr_results, 1):
    print(f"\n[{i}] {doc.page_content[:200]}...")

MMR 검색 결과:

[1] Consumer) 사업부는 미국 내 다음과 같은 수의 소매점을 통해 제품을 판매합니다. 
미국 소매점 수 
NIKE 브랜드 팩토리 스토어 213  
NIKE 브랜드 인라인 매장(사원 전용 매장 포함) 74  
컨버스 점포(팩토리 점포 포함) 82  
합계 369  
미국에는 NIKE 가 8 개의 주요 유통 센터를 보유하고 있습니다. 항목 2 를 참조하십시오...

[2] 목차 
나이키, INC. 
연결재무제표(CONSOLIDATED STATEMENTS OF 
COMPREHENSIVE INCOME) 
연말 5 월 31 일, 
 
(백만 달러) 2023 2022 2021 
순수입 $ 5,070  $ 6,046 $ 5,727  
기타 포괄 소득(손실), 세액 제외:      
순 외화 환산 조정의 변화  267   (522) 4...

[3] 현금 흐름 헤지 
현금흐름헤지상품으로 지정된 파생상품의 공정가치의 모든 변동은 헤지거래의 현금흐름변동에 의해 순이익이 영향을 받을 때까지 누적기타포괄손익(손실)에 기록됩니다. 
효과적인 헤지 결과는 기초 익스포저와 동일한 방식으로 연결 손익계산서에 분류됩니다. 예측된 헤지 거래가 처음 식별된 기간 내에 발생할 가능성이 더 이상 없는 경우, 헤지 
회계는 중...


## 7. 실전 예제: 검색 엔진 구축

In [12]:
class SemanticSearchEngine:
    """시맨틱 검색 엔진 클래스"""
    
    def __init__(self, vector_store, retriever):
        self.vector_store = vector_store
        self.retriever = retriever
    
    def search(self, query: str, k: int = 3):
        """검색 수행"""
        results = self.retriever.invoke(query)
        return results[:k]
    
    def search_with_scores(self, query: str, k: int = 3):
        """유사도 점수와 함께 검색"""
        return self.vector_store.similarity_search_with_score(query, k=k)

# 검색 엔진 인스턴스 생성
search_engine = SemanticSearchEngine(vector_store, retriever)

# 검색 테스트
test_queries = [
    "나이키의 재무 성과",
    "나이키의 주요 제품",
    "나이키의 글로벌 전략"
]

print("=" * 80)
print("검색 엔진 테스트")
print("=" * 80)

for query in test_queries:
    print(f"\n\n[검색어]: {query}")
    print("-" * 80)
    results = search_engine.search(query, k=2)
    for i, doc in enumerate(results, 1):
        print(f"\n[결과 {i}]")
        print(f"출처: {doc.metadata.get('source', 'N/A')}")
        print(f"내용: {doc.page_content[:250]}...")

검색 엔진 테스트


[검색어]: 나이키의 재무 성과
--------------------------------------------------------------------------------

[결과 1]
출처: ./example_data/nke-10k-2023_korean.pdf
내용: 목차 
나이키, INC. 
연결재무제표(CONSOLIDATED STATEMENTS OF 
COMPREHENSIVE INCOME) 
연말 5 월 31 일, 
 
(백만 달러) 2023 2022 2021 
순수입 $ 5,070  $ 6,046 $ 5,727  
기타 포괄 소득(손실), 세액 제외:      
순 외화 환산 조정의 변화  267   (522) 496  
현금 흐름 헤지에 대한 순이익(손실)의 변화  (348)  1,214  (825)...

[결과 2]
출처: ./example_data/nke-10k-2023_korean.pdf
내용: 목차 
항목 8. 재무제표 및 보충 데이터 
NIKE, Inc.의 경영진은 이 연례 보고서에 포함된 정보 및 진술에 대한 책임이 있습니다. 재무제표는 미국에서 일반적으로 인정되는 회계 원칙("U.S. GAAP")에 따라 
작성되었으며 당사의 최선의 추정 및 판단에 근거한 특정 금액을 포함합니다. 이 연례 보고서의 다른 재무 정보는 이러한 재무제표와 일치합니다. 
당사의 회계 시스템에는 자산이 무단 사용 또는 처분으로부터 보호되고 US GAAP ...


[검색어]: 나이키의 주요 제품
--------------------------------------------------------------------------------

[결과 1]
출처: ./example_data/nke-10k-2023_korean.pdf
내용: 목차 
2023 양식 10-K 54 
 
나이키, INC. 
연결손익계산서 
연말 5 월 31 일, 
 
(단위: 백만 단위, 주당 데이터 제외) 2023 2022 2021 
수익 $ 51,217  $ 46

## 주요 포인트 정리

1. **문서 로더**: 다양한 소스에서 문서를 로드
2. **텍스트 분할**: 검색 정확도를 위해 적절한 크기로 분할
3. **임베딩**: 텍스트를 벡터로 변환하여 의미 기반 검색 가능
4. **벡터 스토어**: 임베딩된 문서를 저장하고 검색
5. **검색기**: 표준화된 인터페이스로 검색 수행

**다음 단계**: [120_RAG_Agent.py](120_RAG_Agent.py)에서 검색 결과를 LLM에 통합하는 RAG 패턴을 학습합니다.